# Phase 4: Fine-Tuning LLM (`Qwen/Qwen2.5-3B-Instruct` + QLoRA)

This notebook fine-tunes **Qwen2.5-3B-Instruct** using 4-bit QLoRA on your enterprise dataset.
It trains the model to answer questions strictly from provided passages with inline citations (`[Source: URL]`) and zero hallucination.

### Features:
- 4-bit NF4 Quantization (runs easily on free Kaggle T4 16GB GPU)
- Full checkpointing with automatic resumption (`resume_from_checkpoint=True`)
- Saves lightweight LoRA adapter (~40MB) ready for Antigravity Studio


In [ ]:
# 1. Install QLoRA requirements
!pip install -q transformers peft bitsandbytes trl accelerate datasets
!nvidia-smi


In [ ]:
# 2. Imports and Configuration
import os
import json
import glob
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
DATASET_DIR = '/kaggle/input/enterprise-rag-dataset'
CHECKPOINT_DIR = '/kaggle/working/checkpoints_llm'
FINAL_OUTPUT_DIR = '/kaggle/working/phase4_llm_adapter'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)


In [ ]:
# 3. Load Dataset & Construct Grounded Prompt Training Set
doc_map = {}
chunks_path = os.path.join(DATASET_DIR, 'chunks.jsonl')
if not os.path.exists(chunks_path):
    found = glob.glob(f'{DATASET_DIR}/**/chunks.jsonl', recursive=True)
    if found: chunks_path = found[0]

with open(chunks_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        c = json.loads(line)
        cid = c.get('chunk_id')
        did = c.get('document_id')
        if cid: doc_map[cid] = c
        if did and did not in doc_map: doc_map[did] = c

questions_path = os.path.join(DATASET_DIR, 'questions.jsonl')
if not os.path.exists(questions_path):
    found = glob.glob(f'{DATASET_DIR}/**/questions.jsonl', recursive=True)
    if found: questions_path = found[0]

formatted_training_data = []
SYSTEM_PROMPT = 'You are a precise enterprise knowledge assistant. Answer strictly based on the provided documentation passages with inline [Source: URL] citations. If insufficient information is provided, state refusal.'

with open(questions_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        item = json.loads(line)
        q = item.get('question')
        ans = item.get('gold_answer', '')
        doc_ids = item.get('expected_doc_ids', [])
        
        passages = []
        for did in doc_ids:
            if did in doc_map:
                doc_obj = doc_map[did]
                passages.append(f"[Source: {doc_obj.get('url', 'docs')}]\n{doc_obj.get('text', '')}")
        
        context_text = '\n\n---\n\n'.join(passages) if passages else 'No specific passages retrieved.'
        
        user_prompt = f"DOCUMENTATION PASSAGES:\n{context_text}\n\nUSER QUESTION:\n{q}\n\nTECHNICAL ANSWER:"
        
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
            {'role': 'assistant', 'content': ans}
        ]
        formatted_training_data.append({'messages': messages})

print(f'Prepared {len(formatted_training_data):,} conversational training examples.')
train_dataset = Dataset.from_list(formatted_training_data)


In [ ]:
# 4. Load Model in 4-bit NF4 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# Configure LoRA Adapter
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# 5. Setup TrainingArguments with Resumption & Checkpoints
# NOTE: If you already completed 3 epochs and want to train MORE, increase num_train_epochs (e.g. to 5 or 6).
# The trainer will automatically resume from the last checkpoint and train the additional epochs!
existing_checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'), key=os.path.getmtime)
resume_flag = existing_checkpoints[-1] if existing_checkpoints else None
if resume_flag:
    print(f'🔄 Found checkpoint! Resuming training from: {resume_flag}')
else:
    print('Starting fresh training from step 0.')

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    dataset_text_field=None,
    tokenizer=tokenizer,
    args=training_args,
)


In [ ]:
# 6. Execute Training & Save Adapter
print('Starting QLoRA training...')
trainer.train(resume_from_checkpoint=resume_flag)

# Save final trained adapter
trainer.model.save_pretrained(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)
print(f'✅ Phase 4 LLM Adapter fine-tuning complete! Saved to: {FINAL_OUTPUT_DIR}')

# Verify output files
!ls -lh /kaggle/working/phase4_llm_adapter
